In [ ]:

import glob


import xml.etree.ElementTree as ET
import pandas as pd


In [ ]:
dataset_ffn = '/g/data/kl02/jss548/track_scores/titan/bris20251124'
xml_ffn_list = sorted(glob.glob(f'{dataset_ffn}/*.xml'))

['/g/data/kl02/jss548/track_scores/titan/bris20251124/20251124_000700_titan.xml', '/g/data/kl02/jss548/track_scores/titan/bris20251124/20251124_001200_titan.xml', '/g/data/kl02/jss548/track_scores/titan/bris20251124/20251124_001700_titan.xml', '/g/data/kl02/jss548/track_scores/titan/bris20251124/20251124_002200_titan.xml', '/g/data/kl02/jss548/track_scores/titan/bris20251124/20251124_002700_titan.xml', '/g/data/kl02/jss548/track_scores/titan/bris20251124/20251124_003200_titan.xml', '/g/data/kl02/jss548/track_scores/titan/bris20251124/20251124_003700_titan.xml', '/g/data/kl02/jss548/track_scores/titan/bris20251124/20251124_004200_titan.xml', '/g/data/kl02/jss548/track_scores/titan/bris20251124/20251124_004700_titan.xml', '/g/data/kl02/jss548/track_scores/titan/bris20251124/20251124_005200_titan.xml', '/g/data/kl02/jss548/track_scores/titan/bris20251124/20251124_005700_titan.xml', '/g/data/kl02/jss548/track_scores/titan/bris20251124/20251124_010200_titan.xml', '/g/data/kl02/jss548/track_

In [39]:
#init schema
NS = {"w": "https://reg.bom.gov.au/schema/WxML"}

rows = []
for xml_ffn in xml_ffn_list:
    tree = ET.parse(xml_ffn)
    root = tree.getroot()
    for event in root.findall(".//w:event", NS):
        track_id = event.get("ID")
        for case in event.findall("w:case", NS):
            #skip if case_type is "historical" or "nowcast"
            if case.get("description") in ["history", "forecast"]:
                continue
            #init a row using the track id
            row = {"track_id": track_id}
            # add the timestamp
            time_el = case.find("w:time", NS)
            row["timestamp"] = time_el.text if time_el is not None else None

            # Centroid from ellipse moving-point
            mp = case.find(".//w:ellipse/w:moving-point", NS)
            if mp is not None:
                row["lat"] = float(mp.find("w:latitude", NS).text)
                row["lon"] = float(mp.find("w:longitude", NS).text)
            else:
                continue  # skip


            # Nowcast parameters
            area = case.find("w:nowcast-parameters/w:projected_area", NS)
            if area is not None:
                row["area"] = float(area.text)
            else:
                continue  # skip

            rows.append(row)

df = pd.DataFrame(rows)

# Optional: parse time column as datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])



In [40]:
#select only rows where the track if is 110
df_110 = df[df['track_id'] == '110']
print(df_110)


    track_id           timestamp      lat       lon      area
0        110 2025-11-24 00:07:00 -27.1104  154.9347  698.6250
13       110 2025-11-24 00:12:00 -27.0807  154.9605  659.8120
28       110 2025-11-24 00:17:00 -27.0456  154.9649  594.5620
41       110 2025-11-24 00:22:00 -27.0210  154.9743  510.1880
51       110 2025-11-24 00:27:00 -27.0015  154.9882  407.8120
63       110 2025-11-24 00:32:00 -26.9710  154.9987  294.1880
71       110 2025-11-24 00:37:00 -26.9507  154.9974  191.8120
83       110 2025-11-24 00:42:00 -26.9438  154.9900  174.9380
96       110 2025-11-24 00:47:00 -26.9203  155.0181   65.8125
105      110 2025-11-24 00:52:00 -26.8997  155.0206   30.3750
